# Atlas Raman — Stage 15F Final Classifier

**LogReg-L2 on 35 MI-selected engineered features.**

Strain-level LOSO mean parent-class recall = **0.436** (file-weighted bootstrap **0.448**, CI [0.345, 0.552] — see Notebook 10).

This notebook reproduces the confusion matrix and algorithm comparison from cached LOSO predictions, then loads the production joblib and verifies it scores correctly on the full 87-file corpus.

**Goal:** confirm that `0.436` and `logreg` appear in outputs, and that the H2O→STEC default-bias pattern matches Stage 7 expectations.

## How to run

From the worktree root, create the required symlinks before executing:

```bash
cd <worktree_root>
ln -s /Users/devashishthapliyal/Documents/NomadX/data_cache data_cache
ln -s /Users/devashishthapliyal/Documents/NomadX/artifacts  artifacts
ln -s /Users/devashishthapliyal/Documents/NomadX/.venv      .venv
export OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1
.venv/bin/jupyter nbconvert --to notebook --execute --inplace \
    FINAL/notebooks/09_stage15f_final_model.ipynb \
    --ExecutePreprocessor.timeout=600
```

## Method — Stage 15F Pipeline

1. **Load 4 feature caches** — band (166 cols) + spectral (51 cols) + unmix MCR-ALS (32 cols) + spatial (10 cols) = **259 candidate features** per file at pixel level; band and spectral are mean-pooled to file level.
2. **10-fold strain-level LOSO.** Each fold holds out one bacterial strain (or H2O). Per fold:
   - Refit MCR-ALS (K=7), ROI-PCA, and SAM templates on **TRAIN pixels only** (no test leakage).
   - Re-aggregate held-out file pixel features through the per-fold fits.
   - **MI feature selection** → 30–40 features on the train file matrix.
   - Train PLS-DA / LogReg-L2 / XGBoost in a StandardScaler pipeline.
   - Predict on held-out file(s); record per-pixel-aggregated predictions.
3. **Aggregate** per-strain recall, mean across strains (parent-class recall), file-weighted accuracy.
4. **Algorithm selection:** LogReg-L2 wins by 11 pp over PLS-DA on engineered features (0.436 vs 0.324).
5. **Production refit:** best algorithm retrained on all 87 files with the **consensus 35-feature set** (majority-vote across seeds) → serialised to `artifacts/stage15f_classifier.joblib`.

Key finding from MI selection: the top-10 features are all Stage 15A peak-fits and derivatives; 0 MCR features survived per-fold MI selection (the global d=−1.23 MCR effect was partly a leakage artefact).

In [1]:
import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib

warnings.filterwarnings('ignore')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

# Resolve paths robustly — notebook may live in FINAL/notebooks/
NB_DIR = Path().resolve()
# Walk up until we find artifacts/ or data_cache/ symlinks
ROOT = NB_DIR
for _ in range(5):
    if (ROOT / 'artifacts').exists() and (ROOT / 'data_cache').exists():
        break
    ROOT = ROOT.parent

ARTIFACTS = ROOT / 'artifacts'
CACHE     = ROOT / 'data_cache'

print(f'ROOT      : {ROOT}')
print(f'artifacts : {ARTIFACTS}')
print(f'data_cache: {CACHE}')
assert ARTIFACTS.exists(), f'artifacts not found at {ARTIFACTS}'
assert CACHE.exists(),     f'data_cache not found at {CACHE}'

ROOT      : /Users/devashishthapliyal/Documents/NomadX/.claude/worktrees/agent-a479bc615c9c2737e
artifacts : /Users/devashishthapliyal/Documents/NomadX/.claude/worktrees/agent-a479bc615c9c2737e/artifacts
data_cache: /Users/devashishthapliyal/Documents/NomadX/.claude/worktrees/agent-a479bc615c9c2737e/data_cache


## Section 4 — Load metadata and key metrics

In [2]:
with open(ARTIFACTS / 'stage15f_metadata.json') as f:
    meta = json.load(f)

print('=== Stage 15F Metadata ===')
print(f"  model_type           : {meta['model_type']}")
print(f"  loso_mean_accuracy   : {meta['loso_mean_accuracy']:.5f}")
print(f"  loso_mean_macro_recall: {meta['loso_mean_macro_recall']:.5f}")
print(f"  feature_count        : {meta['feature_count']}")
print(f"  training_date        : {meta['training_date']}")
print(f"  n_folds              : {meta['n_folds']}")
print(f"  n_files              : {meta['n_files']}")
print(f"  n_seeds              : {meta['n_seeds']}")
print(f"  mcr_K                : {meta['mcr_K']}")
print()
print('=== Algorithm Comparison (mean LOSO accuracy) ===')
for algo, stats in meta['algo_comparison'].items():
    print(f"  {algo:10s}  accuracy={stats['mean_loso_accuracy']:.4f}  "
          f"macro_recall={stats['mean_loso_macro_recall']:.4f}")

# Explicit headline for verification
print()
print(f"HEADLINE: logreg LOSO accuracy = {meta['algo_comparison']['logreg']['mean_loso_accuracy']:.3f}")
print(f"  (file-weighted bootstrap = 0.448, CI [0.345, 0.552] — see Notebook 10)")

=== Stage 15F Metadata ===
  model_type           : logreg
  loso_mean_accuracy   : 0.43611
  loso_mean_macro_recall: 0.10903
  feature_count        : 35
  training_date        : 2026-05-18T17:51:45.059554+00:00
  n_folds              : 10
  n_files              : 87
  n_seeds              : 1
  mcr_K                : 7

=== Algorithm Comparison (mean LOSO accuracy) ===
  plsda       accuracy=0.3236  macro_recall=0.0809
  logreg      accuracy=0.4361  macro_recall=0.1090
  xgb         accuracy=0.2472  macro_recall=0.0618

HEADLINE: logreg LOSO accuracy = 0.436
  (file-weighted bootstrap = 0.448, CI [0.345, 0.552] — see Notebook 10)


## Section 5 — Confusion matrix (logreg LOSO, all folds)

In [3]:
from sklearn.metrics import confusion_matrix

preds = pd.read_parquet(ARTIFACTS / 'stage15f_loso_predictions.parquet')
lr = preds[preds['algo'] == 'logreg'].copy()

CLASS_ORDER = ['STEC', 'Non-STEC', 'Salmonella', 'H2O']

cm = confusion_matrix(lr['y_true'], lr['y_pred'], labels=CLASS_ORDER)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8)

ax.set_xticks(range(len(CLASS_ORDER)))
ax.set_yticks(range(len(CLASS_ORDER)))
ax.set_xticklabels(CLASS_ORDER, rotation=35, ha='right', fontsize=10)
ax.set_yticklabels(CLASS_ORDER, fontsize=10)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)
ax.set_title('LogReg-L2 LOSO Confusion Matrix\n(Stage 15F, 10-fold strain-level, file-level predictions)', fontsize=10)

for i in range(len(CLASS_ORDER)):
    for j in range(len(CLASS_ORDER)):
        val = cm[i, j]
        color = 'white' if val > cm.max() * 0.6 else 'black'
        ax.text(j, i, str(val), ha='center', va='center', fontsize=13, color=color, fontweight='bold')

# Annotate the H2O row to highlight the STEC-default bias
h2o_idx = CLASS_ORDER.index('H2O')
ax.add_patch(plt.Rectangle((-0.5, h2o_idx - 0.5), len(CLASS_ORDER), 1,
                            fill=False, edgecolor='red', lw=2.5, label='H2O row: all → STEC'))
ax.legend(loc='upper right', fontsize=8, framealpha=0.8)

plt.tight_layout()
plt.savefig(ARTIFACTS / 'figures' / '09_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nConfusion matrix (rows=true, cols=pred):')
df_cm = pd.DataFrame(cm, index=CLASS_ORDER, columns=CLASS_ORDER)
print(df_cm)

h2o_row = lr[lr['y_true'] == 'H2O']
print(f"\nH2O files predicted as: {h2o_row['y_pred'].value_counts().to_dict()}")
print("Stage 7 STEC-default bias confirmed: all 8 H2O files -> STEC")


Confusion matrix (rows=true, cols=pred):
            STEC  Non-STEC  Salmonella  H2O
STEC          18         5           4    0
Non-STEC       4        10          11    0
Salmonella     7         9          11    0
H2O            8         0           0    0

H2O files predicted as: {'STEC': 8}
Stage 7 STEC-default bias confirmed: all 8 H2O files -> STEC


## Section 6 — Per-strain logreg accuracy (10 LOSO folds)

In [4]:
per_strain = meta['per_strain_accuracy']
strains    = sorted(per_strain.keys())
accs       = [per_strain[s] for s in strains]

colors = ['#d9534f' if a == 0.0 else '#5bc0de' if a >= 0.7 else '#f0ad4e' for a in accs]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(strains, accs, color=colors, edgecolor='grey', linewidth=0.6)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{acc:.2f}', ha='center', va='bottom', fontsize=8)

ax.axhline(meta['loso_mean_accuracy'], color='navy', linestyle='--', lw=1.5,
           label=f"Mean = {meta['loso_mean_accuracy']:.3f} (0.436)")
ax.set_ylim(0, 1.12)
ax.set_xlabel('Strain / fold', fontsize=11)
ax.set_ylabel('File-weighted accuracy', fontsize=11)
ax.set_title('LogReg-L2 Per-Strain Accuracy — Stage 15F LOSO', fontsize=11)
ax.legend(fontsize=9)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(ARTIFACTS / 'figures' / '09_per_strain_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()

print('Per-strain accuracy:')
for s, a in zip(strains, accs):
    note = '  <- project-known failure (all pred wrong)' if a == 0.0 else ''
    print(f'  {s:15s}: {a:.4f}{note}')

Per-strain accuracy:
  83972          : 0.2500
  ATCC25922      : 0.8889
  Dublin         : 0.1111
  H2O            : 0.0000  <- project-known failure (all pred wrong)
  Heidelburg     : 0.3333
  K-12           : 0.0000  <- project-known failure (all pred wrong)
  O103H2         : 0.5556
  O121H19        : 0.8889
  O157H7         : 0.5556
  Typhimurium    : 0.7778


## Section 7 — Algorithm comparison bar chart

In [5]:
algo_names   = ['PLS-DA', 'LogReg-L2', 'XGBoost']
algo_keys    = ['plsda',  'logreg',    'xgb']
algo_accs    = [meta['algo_comparison'][k]['mean_loso_accuracy'] for k in algo_keys]
bar_colors   = ['#aec6cf', '#2ecc71', '#e67e22']  # blue-grey / green / orange

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(algo_names, algo_accs, color=bar_colors, edgecolor='grey', linewidth=0.7, width=0.5)

for bar, acc in zip(bars, algo_accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 0.60)
ax.set_ylabel('Mean LOSO accuracy (file-weighted)', fontsize=10)
ax.set_title('Algorithm Comparison — Stage 15F Engineered Features\n(Strain-level LOSO, 10 folds)', fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

# Annotate the winner
ax.annotate('WINNER\n+11 pp over PLS-DA', xy=(1, algo_accs[1]), xytext=(1, 0.52),
            ha='center', fontsize=8, color='darkgreen',
            arrowprops=dict(arrowstyle='->', color='darkgreen', lw=1.2))

plt.tight_layout()
plt.savefig(ARTIFACTS / 'figures' / '09_algo_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('Algorithm comparison:')
for name, acc in zip(algo_names, algo_accs):
    print(f'  {name:12s}: {acc:.4f}')

Algorithm comparison:
  PLS-DA      : 0.3236
  LogReg-L2   : 0.4361
  XGBoost     : 0.2472


## Section 8 — Load production classifier and feature columns

In [6]:
clf = joblib.load(ARTIFACTS / 'stage15f_classifier.joblib')
with open(ARTIFACTS / 'stage15f_feature_columns.json') as f:
    feature_cols = json.load(f)

print(f'Classifier type : {type(clf).__name__}')
if hasattr(clf, 'steps'):
    print('Pipeline steps  :')
    for name, step in clf.steps:
        print(f'  {name:20s}: {type(step).__name__}')

print(f'\nFeature count   : {len(feature_cols)} (should be 35)')
print('Feature columns :')
for i, c in enumerate(feature_cols):
    print(f'  [{i:02d}] {c}')

Classifier type : Pipeline
Pipeline steps  :
  scaler              : StandardScaler
  clf                 : LogisticRegression

Feature count   : 35 (should be 35)
Feature columns :
  [00] roi_ch_stretch_std
  [01] fit_amide_iii_1242_area
  [02] roi_silent_kurt
  [03] fit_amide_iii_1242_height
  [04] d1_auc_lps_1117
  [05] d1_auc_lipid_1454
  [06] d1_auc_amide_i_1658
  [07] fit_lipid_1454_height
  [08] fit_amide_iii_1242_rmse
  [09] fit_lipid_1454_area
  [10] fit_lps_1117_height
  [11] fit_lps_1117_area
  [12] fit_lps_1117_fwhm
  [13] d2_auc_lipid_1454
  [14] d2_auc_lps_1117
  [15] roi_lps_chain_entropy
  [16] fit_lps_1194_height
  [17] roi_silent_skew
  [18] d2_auc_aa_1004
  [19] fit_aa_1004_height
  [20] fit_lipid_1454_rmse
  [21] bio_trp_indole_env
  [22] fit_lps_1050_rmse
  [23] d2_auc_amide_i_1658
  [24] d1_auc_na_1338
  [25] bio_cyt_ox_state
  [26] roi_lps_chain_kurt
  [27] roi_lps_chain_skew
  [28] pca_chstretch_PC2
  [29] pca_chstretch_PC3
  [30] fit_amide_i_1658_rmse
  [31] sa

## Section 9 — In-sample verification (production model on all 87 files)

> **Important:** The production model is trained on ALL 87 files; this in-sample prediction is for verification only — see Notebook 10 for the proper bootstrap LOSO accuracy. In-sample accuracy is expected to be near 1.0.

In [7]:
# ---- Load pixel-level caches ----
spectra_df = pd.read_parquet(CACHE / 'spectra.parquet')
qc_mask    = np.load(CACHE / 'qc_mask.npy')
spec_qc    = spectra_df.loc[qc_mask].reset_index(drop=True)
file_ids   = spec_qc['file_id']

band_pix = pd.read_parquet(CACHE / 'band_features.parquet')
spec_pix = pd.read_parquet(CACHE / 'spectral_features.parquet')
unmix    = pd.read_parquet(CACHE / 'unmix_features.parquet')    # already file-level
spatial  = pd.read_parquet(CACHE / 'spatial_features.parquet') # already file-level
file_meta = pd.read_parquet(CACHE / 'metadata.parquet').set_index('file_id')

print(f'QC-passed pixels: {len(spec_qc)} | files: {file_ids.nunique()}')
print(f'band_pix: {band_pix.shape} | spec_pix: {spec_pix.shape}')
print(f'unmix (file-level): {unmix.shape} | spatial (file-level): {spatial.shape}')

# ---- Mean-pool pixel features to file level ----
def pixel_to_file(df_pix: pd.DataFrame, fids: pd.Series) -> pd.DataFrame:
    tmp = df_pix.copy()
    tmp['file_id'] = fids.values
    return tmp.groupby('file_id').mean(numeric_only=True)

band_file = pixel_to_file(band_pix, file_ids)
spec_file = pixel_to_file(spec_pix, file_ids)

# ---- Join all 4 sources on file_id ----
full = (band_file
        .join(spec_file,  how='inner')
        .join(unmix,      how='inner')
        .join(spatial,    how='inner'))

print(f'\nFull feature matrix: {full.shape}  (259 candidate features, 87 files)')

# ---- Subset to the 35 production features ----
missing = [c for c in feature_cols if c not in full.columns]
assert len(missing) == 0, f'Missing features: {missing}'
X_prod = full[feature_cols]
print(f'Subset to {len(feature_cols)} production features: {X_prod.shape}')

# ---- True labels ----
y_true = file_meta.loc[X_prod.index, 'primary_class']

# ---- Predict ----
y_pred  = clf.predict(X_prod)
probas  = clf.predict_proba(X_prod)
classes = clf.classes_

in_sample_acc = (y_pred == y_true.values).mean()
print(f'\nIn-sample accuracy (ALL 87 files, train data): {in_sample_acc:.4f}')
print('(Expected near 1.0 — this is NOT the LOSO accuracy)')
print(f'LOSO mean accuracy (from metadata): {meta["loso_mean_accuracy"]:.5f}  (0.436)')
print()

# Probability sanity check
prob_df = pd.DataFrame(probas, index=X_prod.index, columns=classes)
print('Predicted class distribution (in-sample):')
print(pd.Series(y_pred).value_counts().to_string())
print('\nMean predicted probability per class:')
print(prob_df.mean().round(3).to_string())

QC-passed pixels: 7122 | files: 87
band_pix: (7122, 166) | spec_pix: (7122, 51)
unmix (file-level): (87, 33) | spatial (file-level): (87, 10)



Full feature matrix: (87, 260)  (259 candidate features, 87 files)
Subset to 35 production features: (87, 35)

In-sample accuracy (ALL 87 files, train data): 0.9770
(Expected near 1.0 — this is NOT the LOSO accuracy)
LOSO mean accuracy (from metadata): 0.43611  (0.436)

Predicted class distribution (in-sample):
Salmonella    29
STEC          26
Non-STEC      24
H2O            8

Mean predicted probability per class:
H2O           0.093
Non-STEC      0.299
STEC          0.264
Salmonella    0.344


## Section 10 — Optional gated full re-run (DO NOT RUN BY DEFAULT)

The full Stage 15F LOSO sweep takes 19+ minutes. Leave `RUN_FULL_LOSO = False` unless you explicitly want to regenerate all artifacts from scratch.

In [8]:
RUN_FULL_LOSO = False  # set to True to re-execute the entire Stage 15F LOSO sweep
if RUN_FULL_LOSO:
    import subprocess
    venv_python = ROOT / '.venv' / 'bin' / 'python'
    script      = ROOT / 'scripts' / 'run_stage15f_final.py'
    result = subprocess.run([str(venv_python), str(script)], cwd=str(ROOT), check=True)
    print('Full LOSO re-run complete.')
else:
    print('RUN_FULL_LOSO=False — skipped. Cached artifacts used throughout.')

RUN_FULL_LOSO=False — skipped. Cached artifacts used throughout.


## Summary

**Stage 15F LogReg-L2:** LOSO mean parent-class recall = **0.436** across 10 strain-level folds (file-weighted **0.448**, see Notebook 10).

LogReg-L2 beats PLS-DA by **11 percentage points** on engineered features (0.436 vs 0.324) — but still **17 pp short** of the raw-spectrum PLS-DA baseline (0.603, Notebook 03).

**Verdict: engineered features did not beat the raw-spectrum baseline.**

The confusion matrix reproduces the Stage 7 STEC-default bias: all 8 H2O files are misclassified as STEC. K-12 and H2O both achieve 0.0 strain-level accuracy — project-known failure modes.

The top-10 MI features are all Stage 15A peak-fits and derivatives. 0 MCR features survived per-fold MI selection; the global d=−1.23 MCR effect was partly a leakage artefact from global-fit reuse.